# Pipeline EHBG-FACS · 06 · Comparación y validación estadística

**Reúne los resultados de los 5 paradigmas y aplica ANOVA / Friedman / Wilcoxon.**

Carga todos los `*_metrics.csv` escritos por los notebooks 01–05 (misma raíz de Drive), construye la tabla comparativa y el tradeoff costo–factibilidad–flota, y contrasta las diferencias con **rigor estadístico** (Fase 4 del anteproyecto): verificación de supuestos (Shapiro, Levene), ANOVA cuando se cumplen y Friedman + post-hoc Wilcoxon (corrección de Holm) cuando no, dada la cola pesada de los costos.

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50, 100, 200, 300]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = 5   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Cargar todos los resultados

In [ ]:
df = runner.load_all_results(env)
print("filas:", len(df), "| solvers:", sorted(df.solver.unique()))
df.head()

## Tabla resumen (leaderboard)
Promedio sobre todo el banco; ordenado por costo total esperado con recurso.

In [ ]:
display(metrics.leaderboard(df, by="expected_total"))
display(metrics.aggregate_by_size(df))

## Figuras comparativas
Barras por tamaño y tradeoff costo–factibilidad–flota (la región ideal es arriba-izquierda: bajo costo y alta factibilidad).

In [ ]:
import matplotlib.pyplot as plt
viz.plot_comparison(df); plt.show()
for s in sorted(df["size"].unique()):
    viz.plot_tradeoff(df, size=int(s)); plt.show()
viz.save_all_comparison(df, env)

## Validación estadística
Para cada métrica clave y cada tamaño: supuestos, prueba ómnibus (ANOVA/Friedman) y post-hoc Wilcoxon pareado (Holm). Diseño de **bloques por instancia** (mismo ξ por CRN).

In [ ]:
for metric in ["expected_total", "cvar", "feasibility"]:
    for s in sorted(df["size"].unique()):
        cmp = metrics.compare_solvers(df, metric=metric, size=int(s), alpha=proto.significance)
        print("="*70)
        print(metrics.summarize_comparison(cmp))

## Guardar resumen
Escribe la tabla y el resumen estadístico en `results/cross/`.

In [ ]:
import json, pandas as pd
out = env.paths.results / "cross"; out.mkdir(parents=True, exist_ok=True)
metrics.leaderboard(df).to_csv(out / "leaderboard.csv", index=False)
metrics.aggregate_by_size(df).to_csv(out / "aggregate_by_size.csv", index=False)
stats = {f"{m}_n{int(s)}": metrics.compare_solvers(df, metric=m, size=int(s))
         for m in ["expected_total","cvar","feasibility"] for s in sorted(df["size"].unique())}
(out / "statistics.json").write_text(json.dumps(stats, indent=2, default=float))
print("guardado en", out)

**Lectura final.** Si EHBG-FACS se ubica en la región ideal (bajo `E[c]`/`CVaR` con `feasibility` alta) y la diferencia frente a los baselines es **estadísticamente significativa** (p_holm < α) en costo/CVaR a factibilidad comparable, se sostiene la hipótesis general del anteproyecto.